In [1]:
import spacy
nlp = spacy.load("./model/VAERS_SME1SME1_test_v2/model-best")
from spacy.training import Corpus
from spacy.training.example import Example

In [6]:
from tqdm.notebook import tqdm
import os, re, json

In [20]:
data_folder = '../Datasets/VAERS'
res = []
for file in tqdm(os.listdir(data_folder)):
    if file.endswith('.json'):
        
        tmp_data = json.load(open(data_folder+'/'+file,'r'))
        text = tmp_data['pages'][0]
        annotations = tmp_data['annotations']
        human_annos = [x for x in annotations if 'SME' in x['note']]
        llm_annos = [x for x in annotations if x['note'] == 'LLM']

        for anno in human_annos:
            res.append([file, 'Human', anno['textContext']['start'], anno['textContext']['end'], anno['label'], anno['textContext']['text']])

        for anno in llm_annos:
            res.append([file, 'LLM', anno['textContext']['start'], anno['textContext']['end'], anno['label'], anno['textContext']['text']])

        curr_start = 0
        chunks_len = len(text) // 800
        for chunk in range(chunks_len+1):
            curr_text = text[chunk*800:(chunk+1)*800]
            # if the text parts was too long, only analyze the first available parts
            bert_res = nlp(curr_text)
            # if not bert_res.ents: continue
            for ent in bert_res.ents:
                start, end, content = ent.start_char+curr_start, ent.end_char+curr_start, ent.text
                if content!=text[start:end]:
                    print('Warning: ', content, text[start:end])
                    break
                res.append([file, 'BERT', start, end, ent.label_, content]) 
            curr_start += len(curr_text)

  0%|          | 0/1001 [00:00<?, ?it/s]

In [23]:
import pandas as pd

total_annotation = pd.DataFrame(res, columns = ['File','Note','Start','End','Label','Text',])
total_annotation.to_excel('VAERS-BERT-WHOLE.xlsx',index=None)

In [19]:
res[-10:]

[['VAERS_170309-1.json',
  'BERT',
  1004,
  1051,
  'Dx',
  'motoric development was consistent with his age'],
 ['VAERS_170309-1.json',
  'BERT',
  1060,
  1121,
  'SYM',
  'mental development was assessed to be similar to a 1 year old'],
 ['VAERS_170309-1.json',
  'BERT',
  1159,
  1182,
  'Status',
  'condition was improving'],
 ['VAERS_170309-1.json', 'BERT', 1200, 1208, 'Lab', 'MRI scan'],
 ['VAERS_170309-1.json', 'BERT', 1344, 1352, 'Dx', 'asphyxia'],
 ['VAERS_170309-1.json', 'BERT', 1383, 1395, 'Dx', 'inflammation'],
 ['VAERS_170309-1.json',
  'BERT',
  1402,
  1445,
  'Dx',
  'general degeneration of the white substance'],
 ['VAERS_170309-1.json', 'BERT', 1519, 1531, 'Dx', 'encephilitis'],
 ['VAERS_170309-1.json',
  'BERT',
  1600,
  1633,
  'Status',
  't was considered as not recovered'],
 ['VAERS_170309-1.json', 'BERT', 1721, 1726, 'VAX', 'MMRII']]